# RBVS Attention U-Net — inference demo

Loads the trained weights and segments one DRIVE test image (PyTorch).
Weights are a GitHub Release asset — run `python scripts/fetch_weights.py` first,
or point at a local `checkpoints/attention_unet.weights.pt`.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / 'src'))
import numpy as np, torch, matplotlib.pyplot as plt
from rbvs import data as D
from rbvs.infer import predict_prob_full
from rbvs.metrics import compute_metrics
from rbvs.model import build_attention_unet


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = build_attention_unet((128,128,1)).to(device)
state = torch.load('../checkpoints/attention_unet.weights.pt', map_location=device, weights_only=False)
model.load_state_dict(state); model.eval()
print('loaded on', device)


In [ ]:
img_fp, msk_fp = D.test_pairs()[0]
raw = D.load_image(img_fp)
gt  = (D.load_mask(msk_fp).squeeze() > 0).astype(np.uint8)
prob = predict_prob_full(raw, model, device, stride=64)
pred = (prob >= 0.5).astype(np.uint8)
acc, iou, f1 = compute_metrics(gt, pred)
print(f'acc={acc:.4f} iou={iou:.4f} f1={f1:.4f}')


In [ ]:
fig, ax = plt.subplots(1,3, figsize=(15,5))
ax[0].imshow(raw, cmap='gray'); ax[0].set_title('input'); ax[0].axis('off')
ax[1].imshow(pred, cmap='gray'); ax[1].set_title('prediction'); ax[1].axis('off')
ax[2].imshow(gt, cmap='gray'); ax[2].set_title('ground truth'); ax[2].axis('off')
plt.tight_layout(); plt.show()
